## Redes Neurais Artificiais 2026.2

- **Disciplina**: Redes Neurais Artificiais 2026.2
- **Professora**: Elloá B. Guedes (ebgcosta@uea.edu.br)  
- **Github**: http://github.com/elloa  
        

Levando em conta a base de dados **_Dry Bean Dataset_**, esta segunda parte do Projeto Prático 2.2 diz respeito à proposição e avaliação de múltiplas redes neurais artificiais do tipo Feedforward Multilayer Perceptron para o problema da classificação multiclasse de variedades de feijão seco a partir de características geométricas extraídas de imagens dos grãos.

## Testando Redes Neurais com os Atributos Numéricos

1. Abra a base de dados em questão
2. Verifique que todos os atributos preditores são numéricos. O único atributo categórico é o alvo (`Class`), que não deve entrar em X
3. Armazene o atributo alvo em uma variável y e os atributos preditores em uma variável X
4. Efetue uma partição holdout 70/30 com o sklearn, distribuindo os exemplos de maneira aleatória e estratificada (`stratify=y`), com `random_state` fixo para reprodutibilidade
5. Efetue o escalonamento dos atributos


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from joblib import Parallel, delayed, parallel_config, cpu_count
from threadpoolctl import threadpool_limits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import classification_report
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score
)

import matplotlib.pyplot as plt

In [ ]:
raiz = Path.cwd()
if raiz.name == "Notebooks":
    raiz = raiz.parent

df = pd.read_excel(raiz / "DryBeanDataset" / "Dry_Bean_Dataset.xlsx")

In [ ]:
df.dtypes

In [ ]:
y = df["Class"]
y.head()

In [ ]:
X = df.drop("Class", axis=1)
X.head()

Reservamos 30% da base como teste final da etapa 3. Os holdouts desta etapa usam os 70% restantes, de modo que o teste final fique fora da seleção das arquiteturas.

In [ ]:
# Separação externa, usada também no notebook da etapa 3.
X_base, X_final, y_base, y_final = train_test_split(
    X, y, test_size=0.3, random_state=64, stratify=y
)

# Holdout da primeira rede, dentro da base de desenvolvimento.
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y_base, test_size=0.3, random_state=64, stratify=y_base
)

### Escalonando os atributos

O treinamento de uma rede neural artificial é mais eficiente quando os valores que lhes são fornecidos como entrada são pequenos e estão em escalas semelhantes, pois isto favorece a convergência. Isto é feito escalonando-se os atributos, mas precisa ser feito de maneira cautelosa, para que informações do conjunto de teste não sejam fornecidas no treinamento.

Há duas estratégias para tal escalonamento: normalização (que leva os valores ao intervalo [0,1]) e padronização (que centraliza os dados em média 0 e desvio padrão 1). Ambas possuem características particulares, vantagens e limitações, como é possível ver aqui: https://www.analyticsvidhya.com/blog/2020/04/feature-scaling-machine-learning-normalization-standardization/

No nosso caso, vamos usar a padronização. Assim, com os atributos preditores do treinamento, isto é, X_train, deve-se subtrair a média e dividir pelo desvio padrão. Para tanto, use o StandardScaler e ajuste-o com os dados do conjunto de treino (fit), transformando-os em seguida (transform). Depois, apenas transforme os dados do conjunto de teste (transform), usando a média e o desvio padrão calculados no treino, sem chamar o fit novamente.

Se todo o conjunto X for utilizado na padronização, a rede neural receberá informações do conjunto de teste por meio da média e do desvio padrão utilizados para preparar os dados de treinamento, o que não é desejável (vazamento de dados, ou *data leakage*).

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

### Continuando

1. Treine uma rede neural multilayer perceptron para este problema com uma única camada oculta de dez neurônios  
    1.1 Utilize a função de ativação ReLU  
    1.2 Utilize o solver Adam    
    1.3 Imprima o passo a passo do treinamento  
    1.4 Utilize o número máximo de épocas igual a 300
2. Com o modelo em questão, após o treinamento, apresente os resultados para o conjunto de teste:  
    2.1 Matriz de confusão  
    2.2 Acurácia  
    2.3 F-Score (F1)  
    2.4 Precisão  
    2.5 Revocação   
3. Repita o treinamento da mesma rede anterior sem imprimir o passo a passo por 10 vezes  
    3.1 Cada uma destas repetições deve ser feita com uma nova partição holdout 70/30 estratificada, e o escalonamento deve ser reajustado com os dados de treino de cada partição  
    3.2 Apresente a média e o desvio padrão da acurácia e do F-Score para o conjunto de teste  
4. Repita por 10 vezes o treinamento desta mesma rede, mas utilizando o otimizador SGD, com as mesmas 10 partições do item anterior  
    4.1 Apresente a média e o desvio padrão da acurácia e do F-Score para o conjunto de teste  
5. Houve influência da escolha do otimizador no desempenho da rede? Justifique com as médias e os desvios padrão obtidos

### **1. Treine uma rede neural multilayer perceptron para este problema com uma única camada oculta de dez neurônios**

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(10,),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=64,
    verbose=True
)

In [ ]:
with threadpool_limits(limits=1, user_api="blas"):
    mlp.fit(X_train, y_train)

### **2. Com o modelo em questão, após o treinamento, apresente os resultados para o conjunto de teste:**

In [ ]:
y_pred = mlp.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=mlp.classes_)
ConfusionMatrixDisplay(cm, display_labels=mlp.classes_).plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Acurácia:  {acc:.4f}")

In [ ]:
f1 = f1_score(y_test, y_pred, average="macro")
print(f"F1:        {f1:.4f}")

In [ ]:
prec = precision_score(y_test, y_pred, average="macro")
print(f"Precisão:  {prec:.4f}")

In [ ]:
rec = recall_score(y_test, y_pred, average="macro")
print(f"Revocação: {rec:.4f}")

### **3. Repita o treinamento da mesma rede anterior sem imprimir o passo a passo por 10 vezes**

In [ ]:
REP = 10
seeds = range(65, 65 + REP)
n_jobs = min(8, cpu_count())
particoes = []

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_base, y_base, test_size=0.3, stratify=y_base, random_state=seed
    )
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    particoes.append((X_tr_s, X_te_s, y_tr, y_te))

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

def avalia_rede(camadas, ativ, solver, n_ep, i):
    X_tr_s, X_te_s, y_tr, y_te = particoes[i]
    rede = MLPClassifier(
        hidden_layer_sizes=camadas, activation=ativ, solver=solver,
        max_iter=n_ep, random_state=i
    )
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always", ConvergenceWarning)
        rede.fit(X_tr_s, y_tr)

    y_pred = rede.predict(X_te_s)
    return {
        "acc": accuracy_score(y_te, y_pred),
        "f1": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "y_te": y_te,
        "y_pred": y_pred,
        "limite_atingido": any(issubclass(a.category, ConvergenceWarning) for a in avisos),
    }

def avalia_configuracao(nome, camadas, ativ, solver, n_ep):
    reps = [avalia_rede(camadas, ativ, solver, n_ep, i) for i in range(REP)]
    return (nome, solver, n_ep), reps

def avalia_arquiteturas(arquiteturas, solvers, epocas):
    with parallel_config(backend="loky", inner_max_num_threads=1):
        resultados = Parallel(n_jobs=n_jobs, verbose=5)(
            delayed(avalia_configuracao)(nome, camadas, ativ, solver, n_ep)
            for nome, (camadas, ativ) in arquiteturas.items()
            for solver in solvers
            for n_ep in epocas
        )
    return dict(resultados)

In [ ]:
resultados_iniciais = avalia_arquiteturas({"Inicial": ((10,), "relu")}, ["adam", "sgd"], [300])
linhas = []
for (_, solver, _), reps in resultados_iniciais.items():
    accs = pd.Series([r["acc"] for r in reps])
    f1s = pd.Series([r["f1"] for r in reps])
    linhas.append({
        "solver": solver,
        "acc_média": accs.mean(), "acc_std": accs.std(ddof=1),
        "f1_média": f1s.mean(), "f1_std": f1s.std(ddof=1),
    })

resumo_inicial = pd.DataFrame(linhas).set_index("solver")
display(resumo_inicial.round(4))
delta = resumo_inicial.loc["adam", "f1_média"] - resumo_inicial.loc["sgd", "f1_média"]
melhor_solver = "Adam" if delta > 0 else "SGD"
if delta == 0:
    print("Os dois solvers empataram em F1 macro médio.")
else:
    print(f"{melhor_solver} teve F1 macro médio {abs(delta):.4f} maior com as mesmas partições.")
print("Os desvios padrão da tabela mostram a variação dos resultados entre as repetições.")

## Discussão

Nos passos anteriores, você avaliou o desempenho de uma única arquitetura de rede neural, com uma única camada oculta de 10 neurônios e função de ativação ReLU. O otimizador utilizado, seja o SGD ou o Adam, é o algoritmo que usa o gradiente do erro (calculado pela retropropagação) para atualizar os pesos da rede. Neste sentido, a escolha do otimizador é um hiperparâmetro: ela é definida antes do treinamento, não é aprendida a partir dos dados e controla como a rede será treinada. Também são hiperparâmetros a quantidade de camadas ocultas, o número de neurônios, a função de ativação, a quantidade de épocas, a taxa de aprendizado inicial, dentre outros. Já os parâmetros da rede são os pesos e os vieses, ajustados durante o treinamento.

Cabe salientar também que você efetuou o treinamento desta rede por 10 vezes e apresentou os resultados em termos de média ± desvio padrão. Lembre-se que em uma rede neural há a inicialização aleatória de pesos e, em consequência, o desempenho está sujeito a uma flutuação estocástica. Além disso, cada repetição utilizou uma nova partição holdout, o que introduz uma segunda fonte de variação. A execução múltipla das repetições reduz a influência de uma boa ou má "sorte" na escolha dos pesos iniciais e da partição, que poderia distorcer a conclusão no caso de uma única execução, e permite estimar a variabilidade do desempenho por meio do desvio padrão.

Você também aprendeu uma estratégia para escalonar os atributos para uma melhor convergência da rede. Utilize-a em todos os treinamentos e testes propostos a seguir.

## Propondo Novas Arquiteturas

Variando a arquitetura da rede (uma ou duas camadas ocultas, com diferentes números de neurônios em cada uma delas, e a função de ativação) e os hiperparâmetros solver (Adam ou SGD) e número máximo de épocas (100, 150 e 200), atenda ao que se pede:

1. Proponha 10 arquiteturas distintas de RNAs para o problema em questão, à sua escolha
2. Avalie cada uma das arquiteturas perante todas as combinações de hiperparâmetros (2 solvers × 3 números de épocas), repetindo cada treinamento/avaliação por 10 vezes  
    2.1 Cada repetição deve utilizar uma nova partição holdout 70/30 estratificada, com o escalonamento ajustado apenas com os dados de treino, e as mesmas 10 partições devem ser usadas em todas as configurações, para que a comparação seja justa  
    2.2 Ao todo, serão 60 configurações (10 arquiteturas × 2 solvers × 3 números de épocas) e 600 treinamentos
3. Como resultado da avaliação, apresente, com base no conjunto de teste:  
    3.1 As 3 melhores configurações (arquitetura + solver + número de épocas) no tocante ao F-Score (média macro) e à acurácia, considerando a média das 10 repetições (± desvio padrão)  
    3.2 Para cada uma dessas 3 configurações, a repetição em que houve o melhor desempenho: apresente a matriz de confusão e ilustre, para cada classe, os valores de VP (verdadeiros positivos), VN (verdadeiros negativos), FP (falsos positivos) e FN (falsos negativos)

1. Proponha 10 arquiteturas distintas de RNAs para o problema em questão, à sua escolha

In [ ]:
arquiteturas = {
    "N1: (10) relu":         ((10,), "relu"),
    "N2: (32) relu":         ((32,), "relu"),
    "N3: (32) tanh":         ((32,), "tanh"),
    "N4: (32) logistic":     ((32,), "logistic"),
    "N5: (64) logistic":     ((64,), "logistic"),
    "N6: (16,8) relu":       ((16, 8), "relu"),
    "N7: (32,16) relu":      ((32, 16), "relu"),
    "N8: (32,16) tanh":      ((32,16), "tanh"),
    "N9: (32,16) logistic":  ((32, 16), "logistic"),
    "N10: (32) identity":    ((32,), "identity"),
}

2. Avalie cada uma das arquiteturas perante todas as combinações de hiperparâmetros (2 solvers × 3 números de épocas), repetindo cada treinamento/avaliação por 10 vezes  

In [ ]:
solvers = ["adam", "sgd"]

In [ ]:
epocas = [100, 150, 200]

2.1 Cada repetição deve utilizar uma nova partição holdout 70/30 estratificada, com o escalonamento ajustado apenas com os dados de treino, e as mesmas 10 partições devem ser usadas em todas as configurações, para que a comparação seja justa 

In [ ]:
print(f"{len(particoes)} partições estratificadas, sementes de {seeds.start} a {seeds.stop - 1}.")

In [ ]:
classes = sorted(y.unique())

In [ ]:
resultados = avalia_arquiteturas(arquiteturas, solvers, epocas)

In [ ]:
def resume_resultados(resultados, arquiteturas, estrategia):
    linhas = []
    for (nome, solver, n_ep), reps in resultados.items():
        camadas, ativ = arquiteturas[nome]
        accs = pd.Series([r["acc"] for r in reps])
        f1s = pd.Series([r["f1"] for r in reps])
        linhas.append({
            "arquitetura": nome, "camadas": camadas, "ativação": ativ,
            "estratégia": estrategia, "solver": solver, "épocas": n_ep,
            "repetições": len(reps),
            "acc_média": accs.mean(), "acc_std": accs.std(ddof=1),
            "f1_média": f1s.mean(), "f1_std": f1s.std(ddof=1),
            "limites_atingidos": sum(r["limite_atingido"] for r in reps),
        })
    return pd.DataFrame(linhas)

df_resultados = resume_resultados(resultados, arquiteturas, "Própria")

3. Como resultado da avaliação, apresente, com base no conjunto de teste:  
    3.1 As 3 melhores configurações (arquitetura + solver + número de épocas) no tocante ao F-Score (média macro) e à acurácia, considerando a média das 10 repetições (± desvio padrão)  

In [ ]:
top_f1 = df_resultados.sort_values("f1_média", ascending=False).head(3)
top_acc = df_resultados.sort_values("acc_média", ascending=False).head(3)

def mostra(top, titulo):
    print(f"\n=== {titulo} ===")
    for _, r in top.iterrows():
        print(f"{r['arquitetura']} | {r['solver']} | {r['épocas']} ép. -> "
              f"F1 = {r['f1_média']:.4f} ± {r['f1_std']:.4f} | "
              f"Acc = {r['acc_média']:.4f} ± {r['acc_std']:.4f}")

mostra(top_f1, "Top 3 por F1 macro")
mostra(top_acc, "Top 3 por acurácia")

3.2 Para cada uma dessas 3 configurações, a repetição em que houve o melhor desempenho: apresente a matriz de confusão e ilustre, para cada classe, os valores de VP (verdadeiros positivos), VN (verdadeiros negativos), FP (falsos positivos) e FN (falsos negativos)

In [ ]:
def detalha(top, titulo, metrica="f1"):
    print(f"\n##### {titulo} #####")
    for _, r in top.iterrows():
        chave = (r["arquitetura"], r["solver"], r["épocas"])
        reps = resultados[chave]
        melhor = max(range(len(reps)), key=lambda k: reps[k][metrica])
        rep = reps[melhor]

        cm = confusion_matrix(rep["y_te"], rep["y_pred"], labels=classes)
        print(f"\n{chave} | melhor repetição = {melhor + 1} "
              f"(F1 = {rep['f1']:.4f}, Acc = {rep['acc']:.4f})")
        print(pd.DataFrame(cm, index=classes, columns=classes))

        total = cm.sum()
        VP = cm.diagonal()
        FP = cm.sum(axis=0) - VP
        FN = cm.sum(axis=1) - VP
        VN = total - VP - FP - FN
        print(pd.DataFrame({"VP": VP, "VN": VN, "FP": FP, "FN": FN}, index=classes))

In [ ]:
detalha(top_f1, "Detalhe: top 3 por F1")
detalha(top_acc, "Detalhe: top 3 por acurácia", metrica="acc")

## Estimando o número de neurônios

Um dos problemas de pesquisa com redes neurais artificiais consiste na determinação do número de neurônios em sua arquitetura. Embora não seja possível definir a priori qual rede neural é adequada para um problema, pois isto só pode ser determinado empiricamente, mediante a avaliação de várias arquiteturas, há regras na literatura que sugerem o número de neurônios escondidos, tal como a regra da Pirâmide Geométrica, dada a seguir:

$$N_h = \alpha \cdot \sqrt{N_i \cdot N_o},$$

em que $N_h$ é o número de neurônios ocultos (a serem distribuídos em uma ou duas camadas ocultas), $N_i$ é o número de neurônios na camada de entrada e $N_o$ é o número de neurônios na camada de saída. Como $N_h$ deve ser um número inteiro, arredonde o resultado para o inteiro mais próximo.

1. Consulte a documentação da classe MLPClassifier (disponível em https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) e obtenha os valores de $N_i$ e $N_o$ para a base _Dry Bean_, considerando o número de atributos preditores e o número de classes do problema.
2. Teste os valores de $\alpha$ como sendo iguais a $0.5$, $2$ e $3$.
3. Proponha pelo menos 30 redes neurais segundo a regra da pirâmide geométrica e teste-as nos mesmos termos estabelecidos anteriormente (solver, épocas, 10 repetições com as mesmas partições holdout, escalonamento, etc.)  
    3.1 Apresente as top-3 melhores redes no tocante à F-Score e Acurácia, com base no conjunto de teste

### Respostas

Na documentação, na seção de atributos, dois itens são úteis para a resposta:  
- n_features_in_: número de features vistas durante o fit.
- n_outputs_: número de neurônios na camada de saída

Logo, são Ni (número de entradas, ou seja, número de atributos preditores) e No (número de saídas, ou seja o número de classes) respectivamente.

In [ ]:
# Testando alphas
Ni, No = 16, 7
alphas = [0.5, 2, 3]

In [ ]:
for alpha in alphas:
    Nh = round(alpha* (Ni * No) ** 0.5)
    print(f"a = {alpha} -> Nh = {Nh}")

A regra da pirâmide geométrica (proposta por Timothy Masters em 1993) é uma heurística empírica usada para definir a quantidade ideal de neurônios nas camadas ocultas de uma rede neural artificial (especialmente em redes Multilayer Perceptron - MLP).

In [ ]:
arq_piramide = {}
for alpha in alphas:
    Nh = round(alpha * (Ni * No) ** 0.5)

    formatos = {
        "1c":      (Nh,),
        "2c 75/25": (round(Nh * 0.75), Nh - round(Nh * 0.75)),
        "2c 50/50": (round(Nh * 0.50), Nh - round(Nh * 0.50)),
        "2c 25/75": (round(Nh * 0.25), Nh - round(Nh * 0.25)),
    }

    for nome_f, camadas in formatos.items():
        for ativ in ["relu", "tanh"]:
            arq_piramide[f"α={alpha} Nh={Nh} {nome_f} {camadas} {ativ}"] = (camadas, ativ)

    arq_piramide[f"α={alpha} Nh={Nh} 1c {formatos['1c']} logistic"] = (formatos["1c"], "logistic")
    arq_piramide[f"α={alpha} Nh={Nh} 2c 50/50 {formatos['2c 50/50']} logistic"] = (formatos["2c 50/50"], "logistic")

In [ ]:
print(len(arq_piramide), "redes") 
for nome, (camadas, ativ) in arq_piramide.items():
    print(nome)

In [ ]:
resultados_p = avalia_arquiteturas(arq_piramide, solvers, epocas)
df_p = resume_resultados(resultados_p, arq_piramide, "Pirâmide Geométrica")

In [ ]:
def show(top, titulo):
    print(f"\n=== {titulo} ===")
    for _, r in top.iterrows():
        print(f"{r['arquitetura']} | {r['solver']} | {r['épocas']} ép. -> "
              f"F1 = {r['f1_média']:.4f} ± {r['f1_std']:.4f} | "
              f"Acc = {r['acc_média']:.4f} ± {r['acc_std']:.4f}")

top_f1_p = df_p.sort_values("f1_média", ascending=False).head(3)
top_acc_p = df_p.sort_values("acc_média", ascending=False).head(3)

show(top_f1_p, "Top 3 por F1 macro")
show(top_acc_p, "Top 3 por acurácia")

## Considerações Parciais

1. É possível identificar uma rede com desempenho superior às demais? 
2. Qual estratégia mostrou-se mais produtiva para a obtenção de boas arquiteturas (Estratégia Própria ou Pirâmide Geométrica)? Por quê?
3. Um número maior de épocas trouxe melhorias? Justifique com os resultados obtidos, considerando também cada um dos solvers (Adam e SGD)
4. Qual a maior dificuldade de resolução do problema proposto perante as RNAs?

In [ ]:
todos = pd.concat([df_resultados, df_p], ignore_index=True)
ranking = todos.sort_values(["f1_média", "acc_média"], ascending=False)
top_6 = ranking.drop_duplicates(["camadas", "ativação"]).head(6)
top_6.to_csv(raiz / "Notebooks" / "top_6_rnas.csv", index=False)

print("Melhor configuração de cada estratégia:")
display(ranking.drop_duplicates("estratégia").round(4))

print("F1 macro médio das configurações por solver e limite de épocas:")
display(todos.groupby(["solver", "épocas"])["f1_média"].mean().unstack().round(4))

print("Seis arquiteturas selecionadas para a etapa 3:")
display(top_6.round(4))

Usamos F1 macro para comparar as estratégias, pois cada variedade tem o mesmo peso na métrica. Na tabela por solver e limite de épocas, cada média considera todas as arquiteturas testadas naquela combinação. A coluna de limites atingidos indica quantas redes chegaram ao limite de épocas com aviso de convergência.